# 06. Adding Margins and Totals in Pandas

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week10/06.Adding-Margins-Totals/notebooks/01_06.Adding-Margins-Totals.ipynb)

## Overview
When preparing reports for stakeholders, summary tables usually require **margins** (row and column totals, plus a grand total). In Pandas, `margins=True` can be enabled in both `pd.pivot_table()` and `pd.crosstab()`.

In this notebook, we cover:
1. **Adding Margins to Pivot Tables**: Using `margins=True` and `margins_name='Total'`.
2. **The Aggregation Nuance**: Why margins compute the statistic over the entire unpartitioned dataset rather than averaging the cell averages.
3. **Contingency Tables with `pd.crosstab()`**: Counting observations across categorical variables.
4. **Normalisation**: Converting raw counts into percentages across rows (`normalize='index'`), columns (`normalize='columns'`), or the grand total (`normalize='all'`).

## 1. Setup: University Scholarships and Enrolment Data

We analyse domestic and international student cohorts across university campuses.

In [ ]:
import pandas as pd
import numpy as np

df_scholarships = pd.DataFrame({
    'campus': [
        'North Sydney', 'North Sydney', 'North Sydney', 'North Sydney',
        'Melbourne', 'Melbourne', 'Melbourne', 'Melbourne',
        'Brisbane', 'Brisbane', 'Brisbane', 'Brisbane'
    ],
    'residency': [
        'Domestic', 'International', 'Domestic', 'International',
        'Domestic', 'International', 'Domestic', 'International',
        'Domestic', 'International', 'Domestic', 'International'
    ],
    'faculty': [
        'Health Sciences', 'Health Sciences', 'Law & Business', 'Law & Business',
        'Health Sciences', 'Health Sciences', 'Education & Arts', 'Education & Arts',
        'Health Sciences', 'Health Sciences', 'Law & Business', 'Law & Business'
    ],
    'enrolments': [450, 150, 380, 220, 520, 180, 410, 90, 320, 110, 290, 140],
    'scholarship_kaud': [45, 60, 30, 85, 50, 75, 25, 40, 35, 55, 20, 65]
})

print("Scholarships Dataset:")
display(df_scholarships)

## 2. Pivot Tables with Margins (`margins=True`)

Setting `margins=True` adds an extra row and column computing the aggregate across all rows and columns.

In [ ]:
# Total enrolments by campus and residency, with margins
enrolment_table = pd.pivot_table(
    df_scholarships,
    values='enrolments',
    index='campus',
    columns='residency',
    aggfunc='sum',
    margins=True,
    margins_name='Total Enrolments'
)

print("Total Enrolments with Margins:")
display(enrolment_table)

## 3. The Aggregation Nuance: True Grand Aggregates

> **Important Rule**: When `aggfunc='mean'`, the bottom-right grand total is **NOT** the mean of the four cell averages! It is the true arithmetic mean of all 12 observations in the underlying dataset.

In [ ]:
mean_table = pd.pivot_table(
    df_scholarships,
    values='scholarship_kaud',
    index='campus',
    columns='residency',
    aggfunc='mean',
    margins=True,
    margins_name='Grand Mean'
)

print("Average Scholarship (kAUD) with True Grand Mean:")
display(mean_table.round(1))
print(f"Underlying dataset true mean: {df_scholarships['scholarship_kaud'].mean():.1f} kAUD")

## 4. Frequency Tables & Percentage Normalisation (`pd.crosstab`)

`pd.crosstab()` computes frequency counts of factors.
- `normalize='index'`: Row percentages sum to 100%.
- `normalize='columns'`: Column percentages sum to 100%.
- `normalize='all'`: All cells sum to 100%.

In [ ]:
# Row-normalised percentages
pct_by_campus = pd.crosstab(
    df_scholarships['campus'],
    df_scholarships['faculty'],
    normalize='index',
    margins=True
) * 100

print("Faculty Offerings Distribution (% by Campus):")
display(pct_by_campus.round(1))

## 5. Practical Exercises

### Exercise 1: National Park Visitors with Margins
Using the national park dataset below:
1. Create a pivot table summing `visitors` across `state` and `activity`.
2. Enable `margins=True` and label the total `Grand Total`.
3. Fill missing combinations with `0`.

In [ ]:
parks = pd.DataFrame({
    'state': ['NSW', 'NSW', 'VIC', 'VIC', 'QLD', 'QLD', 'TAS', 'TAS'],
    'activity': ['Hiking', 'Camping', 'Hiking', 'Camping', 'Hiking', 'Kayaking', 'Hiking', 'Kayaking'],
    'visitors': [12500, 8400, 11200, 9600, 14100, 4200, 6800, 3100]
})

# --- Student Code Here ---
# park_pivot = ...

# --- Solution ---
park_pivot = pd.pivot_table(
    parks,
    values='visitors',
    index='state',
    columns='activity',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Grand Total'
)
display(park_pivot)

### Exercise 2: Activity Distribution Percentages
Use `pd.crosstab()` with `normalize='index'` and `margins=True` to compute the percentage distribution of activities within each state.

In [ ]:
# --- Student Code Here ---
# activity_pct = ...

# --- Solution ---
activity_pct = pd.crosstab(
    parks['state'],
    parks['activity'],
    normalize='index',
    margins=True
) * 100
display(activity_pct.round(1))

## 6. Key Takeaways

1. **`margins=True`**: Automatically adds subtotal rows and columns, as well as a bottom-right grand aggregate.
2. **Correct Unweighted Calculation**: Margins recalculate the function over the raw data, preventing skewed averages of un-equal group sizes.
3. **`pd.crosstab()` Normalisation**: Provides an instant way to turn raw contingency counts into percentages (`'index'`, `'columns'`, or `'all'`).